# 03 — Feature Engineering

Formalise the transformations needed for log-log regression and build a reusable sklearn Pipeline.

**Reference:** Géron Ch.2 — *Prepare the Data for Machine Learning Algorithms*, Ch.4 — *Training Models* (Linear Regression, Normal Equation).

> *"Write functions for all data transformations you apply, for several good reasons: you can easily prepare the data the next time you get a fresh dataset; you can apply these transformations in your future projects; you can clean and prepare the test set; once your solution is live, you can clean new data instances."* — Géron Ch.2

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from price_elasticity.config import load_config
from price_elasticity.data import load_training_frame
from price_elasticity.features import prepare_features

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
df_raw = load_training_frame(config)
df = prepare_features(df_raw.copy(), config)

price_col = config['data']['price_column'].lower()
demand_col = config['data']['demand_column'].lower()
product_col = config['data']['product_column'].lower()

for col in [price_col, demand_col]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

valid = df[(df[price_col] > 0) & (df[demand_col] > 0)].copy()
print(f'Valid observations: {len(valid):,}')
print(f'New columns added by feature engineering: {sorted(set(df.columns) - set(df_raw.columns))}')

## 1. Log Transformation — The Core Transformation

The log-log model requires log-transforming both price and demand:

$$\text{If } Q = A \cdot P^\beta \implies \ln(Q) = \ln(A) + \beta \cdot \ln(P)$$

This converts a non-linear power-law relationship into a linear one — exactly what OLS needs.

In [ ]:
valid['log_price'] = np.log(valid[price_col])
valid['log_demand'] = np.log(valid[demand_col])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(valid[price_col].clip(upper=valid[price_col].quantile(0.99)), bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Price — original scale')
axes[0, 0].set_xlabel('Price ($)')

axes[0, 1].hist(valid['log_price'], bins=50, color='coral', edgecolor='white')
axes[0, 1].set_title('ln(Price) — log scale')
axes[0, 1].set_xlabel('ln(Price)')

axes[1, 0].hist(valid[demand_col].clip(upper=valid[demand_col].quantile(0.99)), bins=50, color='steelblue', edgecolor='white')
axes[1, 0].set_title('Demand — original scale')
axes[1, 0].set_xlabel('Demand')

axes[1, 1].hist(valid['log_demand'], bins=50, color='coral', edgecolor='white')
axes[1, 1].set_title('ln(Demand) — log scale')
axes[1, 1].set_xlabel('ln(Demand)')

plt.suptitle('Log transformation normalises both variables', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Discount Rate Feature

In [ ]:
if 'discount_rate' in valid.columns:
    print('discount_rate statistics:')
    display(valid['discount_rate'].describe().to_frame().T.round(4))
    print(f'\nShare with any discount: {(valid["discount_rate"] > 0).mean():.1%}')
    print(f'Share with > 20% discount: {(valid["discount_rate"] > 0.2).mean():.1%}')
else:
    # Create it manually if needed
    original_price_col = 'price' if 'price' in valid.columns else None
    if original_price_col:
        valid['discount_rate'] = 1 - valid[price_col] / valid[original_price_col].replace(0, np.nan)
        print('discount_rate created from price and disc_price columns')
        display(valid['discount_rate'].describe().to_frame().T.round(4))

## 3. sklearn Pipeline — Reusable Preprocessing

Géron Ch.2 and Ch.4 emphasise building Pipelines that encapsulate all transformations. For the per-product regression, the main pipeline is the log transform + OLS. For any supervised extension, here is how to structure it:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LinearRegression
import numpy as np

# Log transformer: converts positive-valued series to log space
log_transformer = FunctionTransformer(np.log, validate=True)

# Full log-log pipeline for a single product
log_log_pipeline = Pipeline([
    ('log_transform', log_transformer),
    ('ols', LinearRegression()),
])

# Demonstrate on one product
sample_product = valid[product_col].value_counts().idxmax()
prod_df = valid[valid[product_col] == sample_product].copy()

X = prod_df[[price_col]].to_numpy(dtype=float)
y = prod_df[demand_col].to_numpy(dtype=float)

log_log_pipeline.fit(X, np.log(y))  # y also log-transformed

beta = log_log_pipeline.named_steps['ols'].coef_[0]
print(f'Product: {sample_product[:60]}')
print(f'Estimated elasticity (β): {beta:.3f}')
print(f'Interpretation: a 1% price increase → {beta:.2f}% change in demand')

## 4. Feature Summary

| Feature | Source | Transformation | Purpose |
|---------|--------|----------------|---------|
| `log_price` | `disc_price` | `ln(price)` | Linear regressor |
| `log_demand` | `imp_count` | `ln(demand)` | Log-scale target |
| `discount_rate` | `price`, `disc_price` | `1 - disc/full` | Feature for analysis |
| `observed_year/month/day` | `date_imp_d` | Date parts | Temporal features |

**All other columns** (text, URLs, manufacturer, etc.) are excluded from the elasticity model — they would add noise without improving the per-product regression.

**Next:** `04_modeling_and_business_results.ipynb` — run the full per-product OLS, compute confidence intervals, and translate to business decisions.